<div style="background:#03045E;padding:28px 32px;border-radius:16px;font-family:Arial,sans-serif;">
<img src="../assets/jekacode-logo.png" alt="Jekacode" width="240"/>
<p style="color:#16D365;font-size:12px;letter-spacing:2.5px;margin:18px 0 6px 0;">JEKACODE AI ENGINEERING · WEEK 7 · DAY 1</p>
<h1 style="color:#ffffff;margin:0;font-size:28px;">Agents vs chatbots</h1>
<p style="color:#d7deea;margin:10px 0 0 0;font-size:16px;">A tool is just a Python function with a clear name.</p>
</div>


## What we want to achieve

Draw chatbot vs agent. Build one agent + one or two tools. Know that tool choice can be **inconsistent**.

## Tools you need (names only)

Gemini/Grok. Gradio: `python week07-agents/gradio_app.py`. Streamlit: `research_assistant.py`.

## How to (do these before the first code cell if you have not)

1. Bookmark [../guides/HOW_TO.md](../guides/HOW_TO.md) — Ollama install, Gemini key, Grok key, Gradio
2. VS Code on the **course folder** · terminal shows `(.venv)`
3. Kernel = Python inside `.venv`
4. Gemini: [Google AI Studio](https://aistudio.google.com/app/apikey) → `.env` → `GEMINI_API_KEY=`
5. Optional Grok: [console.x.ai](https://console.x.ai) → `GROK_API_KEY=`
6. Optional Ollama: [ollama.com](https://ollama.com) then `ollama run llama3.2`

## What goes on behind the scenes

The model does not magically browse the internet here. **You** write `notes_search()`. The LLM only *picks* or *writes after* the tool. Each extra `ask()` adds **latency**. Behind the scenes an ‘agent’ is a loop: think → maybe call a function → observe → write.

**Keys & installs (bookmark):** [../guides/HOW_TO.md](../guides/HOW_TO.md) · **Terms:** [../guides/AI_ENGINEERING_TERMS.md](../guides/AI_ENGINEERING_TERMS.md)


In [ ]:
# --- Why this cell exists (read once) ---
# Python only finds packages that live on a list of folders called sys.path.
# This notebook sits in a week folder. The jekacode helper lives one folder up.
# Novices: you are not "hacking". You are telling Python where the course lives.

import sys
# sys = the "system" module. We use it to change where Python looks for imports.

from pathlib import Path
# Path is a friendly way to talk about folders. It works on Mac, Windows, and Linux.

root = Path.cwd()
# cwd = current working directory = "the folder this notebook thinks it is in".

if not (root / "jekacode").exists():
    # If we cannot see the jekacode folder here, we are inside week01, week02, ...
    root = root.parent
    # parent = the folder above this one (the course root).

if str(root) not in sys.path:
    sys.path.append(str(root))
    # Now `from jekacode.ai import ask` can succeed.

print("Course folder Python will use:", root)
print("You should see jekacode inside that folder.")


## Theory: three cousins (say this out loud)

| Thing | What it does | Example |
|---|---|---|
| **Chatbot** | One question → one answer | Study assistant Week 4 |
| **Workflow** | Fixed steps you wrote | Classify email, then draft (Week 8) |
| **Agent** | Model *chooses* tools / next step | Research: search notes, then report |

A **tool** is ordinary Python: search a dict, read a file, add numbers. **Function calling** in big products is the model emitting a structured “please run `notes_search` with topic=jamb”. We start simpler: we *ask* it to pick, then *we* run the function. That is honest AI engineering.

**Memory (tiny):** we pass the tool result back in the next prompt. We are not training the model.

Picture: [../visuals/agent-flow.html](../visuals/agent-flow.html)


<div style="border-left:6px solid #16D365;background:#F4F6FB;padding:14px 16px;border-radius:0 8px 8px 0;font-family:Arial,sans-serif;">
<strong style="color:#03045E;">Import story</strong>
<p style="margin:8px 0 0 0;color:#1b1f2a;">`ask` talks to the LLM. Your functions talk to *data*. Keep those jobs separate so grades and fees never get invented.</p>
</div>


In [ ]:
from jekacode.ai import ask
# from package import function — take the named tool out of jekacode/ai.py

# A tiny knowledge tool. No internet. Predictable. Same input → same output.
NOTES = {"jamb": "JAMB is a Nigerian university entrance exam. English is compulsory."}

def notes_search(topic: str) -> str:
    """Look up a keyword. This is deterministic Python, not an LLM."""
    # .lower() so "JAMB" and "jamb" match the same locker.
    t = topic.lower()
    for key, value in NOTES.items():
        # .items() walks (key, value) pairs in the dict.
        if key in t:
            return value
    return "No local note found."

# We ask the model only to CHOOSE. We do not let it invent the JAMB fact.
choice = ask(
    "Goal: 6-line JAMB briefing for a parent. Reply ONLY notes_search:jamb",
    system="You pick tools. Be short. Do not write the briefing yet.",
    provider="gemini",
)
print("model chose:", choice)

# WE run the tool. The model does not.
info = notes_search("jamb")
print("tool returned:", info)

# Second inference: write using the tool result. This is the agent loop, two hops.
print(ask(
    f"Write the briefing.\nTool result: {info}",
    system="Clear. No fluff. Do not add facts that are not in the tool result.",
    provider="gemini",
))
